# 3e. Survival Targets and Stratified Splits
This notebook processes the `interim` cohorts, computes per-cohort median survival for OS/PFS to create binary targets, and generates 10 robust stratified train/test splits.

In [ ]:
# !rm -rf batchcor-rna-embeds/  # if already cloned
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q --upgrade ipython ipykernel


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import sys
from pathlib import Path
from loguru import logger

# Add project root to path if running locally
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# When running in Colab, the root is /content/batchcor-rna-embeds
if "/content/batchcor-rna-embeds" not in sys.path:
    sys.path.insert(0, "/content/batchcor-rna-embeds")

from batchcor_rna_emb.config import *
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.survival_targets import add_survival_targets
from batchcor_rna_emb.split_utils import generate_stratified_splits

In [ ]:
# Source: Google Drive interim data
DATA_INTERIM_DIR = Path('/content/drive/MyDrive/data/interim')

logger.info("DATA_INTERIM_DIR: {}", DATA_INTERIM_DIR)
logger.info("Available cohorts: {}", [p.name for p in sorted(DATA_INTERIM_DIR.glob('*.adata.zarr'))])


In [ ]:
N_SPLITS = 10
TEST_SIZE = 0.2

for zarr_path in sorted(DATA_INTERIM_DIR.glob("*.adata.zarr")):
    cohort_name = zarr_path.name.replace(".adata.zarr", "")
    logger.info("\n" + "=" * 70)
    logger.info("Processing: {}", cohort_name)
    logger.info("=" * 70)

    try:
        adata = load_cohort(zarr_path)
    except Exception as e:
        logger.warning("Failed to load {}, skipping. Error: {}", cohort_name, e)
        continue

    logger.info("Loaded: {} samples x {} genes", adata.n_obs, adata.n_vars)

    # 1. Add Survival Targets
    add_survival_targets(adata)

    # 2. Generate Stratified Splits
    stratify_cols = [
        DIAGNOSIS_COL, 
        BATCH_COL, 
        "Target_OS_bin", 
        "Target_PFS_bin", 
        "Response"
    ]
    generate_stratified_splits(
        adata,
        stratify_cols=stratify_cols,
        n_splits=N_SPLITS,
        test_size=TEST_SIZE
    )

    # 3. Save back to interim
    logger.info("Saving updated cohort back to {}", zarr_path)
    save_adata_zarr(adata, zarr_path)
    logger.info("Saved successfully.")

In [ ]:
print("\n" + "*" * 60)
print("VERIFICATION STEP: Check if binarization and splits are correct")
print("*" * 60)

for zarr_path in sorted(DATA_INTERIM_DIR.glob("*.adata.zarr")):
    cohort_name = zarr_path.name.replace(".adata.zarr", "")
    adata = load_cohort(zarr_path)
    
    print(f"\n=== {cohort_name} ===")
    
    # Check splits
    splits = [c for c in adata.obs.columns if c.startswith("Split_seed_")]
    if splits:
        print(f"✅ Splits generated successfully: {len(splits)} columns (from {splits[0]} to {splits[-1]})")
    else:
        print("❌ WARNING: No Split_seed columns found!")
        
    # Check survival targets
    for target in ['Target_OS_bin', 'Target_PFS_bin']:
        if target in adata.obs.columns:
            val_counts = adata.obs[target].value_counts(dropna=False)
            print(f"✅ {target} successfully created:")
            print(val_counts.to_string())
        else:
            print(f"⚠️ {target} not found (this is normal if cohort doesn't have survival data for it)")
